# โหลด Tool ทั้งหมดที่จะใช้

## การติดตั้ง Dependencies
ในเซลล์แรก เราติดตั้ง libraries ที่จำเป็น:
- **transformers**: สำหรับโหลดและใช้งาน pre-trained models
- **torch**: Deep learning framework ที่ต้องใช้กับ transformers
- **sentencepiece**: Tokenizer library สำหรับการประมวลผลภาษา

## Pipeline ที่ใช้

### 1. **NER Pipeline** (Named Entity Recognition)
ใช้โมเดล `pythainlp/thainer-corpus-v2-base-model` สำหรับ:
- ระบุคน (PERSON)
- ระบุองค์กร (ORGANIZATION)
- ระบุสถานที่ (LOCATION)
- กรองเฉพาะ entities ที่มี confidence score > 0.85

### 2. **Sentiment Analysis** (Rule-based)
วิเคราะห์ความรู้สึกจากข้อความโดย:
- นับคำบ่งชี้ด้านบวก (positive keywords)
- นับคำบ่งชี้ด้านลบ (negative keywords)
- เปรียบเทียบจำนวนเพื่อกำหนด sentiment

## ผลลัพธ์
- **NER Results**: รายชื่อบุคคลและองค์กรที่พบในข้อความ
- **Sentiment**: ความรู้สึก (positive/negative/neutral) พร้อมคะแนน
- **Output**: บันทึก CSV ที่ประมวลผลแล้ว และสรุปข้อมูลสถิติ

In [ ]:
pip install transformers torch sentencepiece

# เทสโมเดล NER thainer-corpus-v2-base-model

In [ ]:
# test_ner.py
from transformers import pipeline

ner = pipeline(
    task='ner',
    model='pythainlp/thainer-corpus-v2-base-model',
    aggregation_strategy='simple'
)

test_text = "ดร.เทดรอส องค์การอนามัยโลก ยืนยันว่าไม่พบการระบาดของไวรัสฮันตา"
result = ner(test_text)
print(result)

# 1_scrape

In [ ]:
# 01_scrape.py
import pandas as pd
import urllib.request
from html.parser import HTMLParser
import re
import os

class SimpleArticleParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_title = False
        self.in_p = False
        self.in_script = False   # เพิ่ม: กัน JS ปน
        self.in_style = False    # เพิ่ม: กัน CSS ปน
        self.title_parts = []
        self.current_paragraph = []
        self.paragraphs = []

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag == "title":
            self.in_title = True
        elif tag == "p":
            self.in_p = True
            self.current_paragraph = []
        elif tag == "script":
            self.in_script = True
        elif tag == "style":
            self.in_style = True

    def handle_endtag(self, tag):
        tag = tag.lower()
        if tag == "title":
            self.in_title = False
        elif tag == "p":
            self.in_p = False
            paragraph = "".join(self.current_paragraph).strip()
            if len(paragraph) > 20:   # กรองย่อหน้าที่สั้นเกินไป
                self.paragraphs.append(paragraph)
            self.current_paragraph = []
        elif tag == "script":
            self.in_script = False
        elif tag == "style":
            self.in_style = False

    def handle_data(self, data):
        if self.in_script or self.in_style:
            return   # ข้าม JS และ CSS
        if self.in_title:
            self.title_parts.append(data)
        elif self.in_p:
            self.current_paragraph.append(data)


def clean_text(text):
    """ลบ whitespace ซ้ำ และบรรทัดว่าง"""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# Header จำเป็นมาก — ไม่งั้น 403
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "th-TH,th;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

urls = [
    "https://www.thairath.co.th/news/foreign/2932377",
    "https://www.thairath.co.th/news/local/bangkok/2932446",
    "https://www.thairath.co.th/news/foreign/2932436",
    "https://www.thairath.co.th/news/local/northeast/2932077",
    "https://www.thairath.co.th/news/politic/2932597",
    "https://www.thairath.co.th/news/politic/2932607",
    "https://www.thairath.co.th/news/local/northeast/2930061#aWQ9NjJkZjcwNmI5OTJmNmUwMDEyZTY5MjMxJnBvcz0zJnJ1bGU9MCZjb250ZW50X3NpdGU9dGhhaXJhdGgtb25saW5l",
    "https://www.thairath.co.th/news/politic/2932578",
    "https://www.thairath.co.th/news/politic/2932602",
    "https://www.thairath.co.th/news/local/central/2932577",
    # เพิ่ม URL ที่ต้องการ scrape ได้ที่นี่
]

os.makedirs("data", exist_ok=True)   # สร้างโฟลเดอร์ถ้ายังไม่มี

records = []
for url in urls:
    try:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=15) as response:
            html = response.read().decode("utf-8", errors="replace")

        parser = SimpleArticleParser()
        parser.feed(html)

        title = " ".join(part.strip() for part in parser.title_parts).strip()
        text = clean_text("\n".join(p.strip() for p in parser.paragraphs))

        # เตือนถ้าข้อความน้อยเกินไป — อาจ scrape ไม่ได้จริง
        word_count = len(text)
        status = "OK" if word_count > 100 else "WARN (text สั้นมาก)"

        records.append({
            "url": url,
            "title": title,
            "text": text,
            "date": "",
            "char_count": word_count
        })
        print(f"{status}: {title[:50]} [{word_count} chars]")

    except Exception as e:
        print(f"SKIP: {url} — {e}")

df = pd.DataFrame(records)
df.to_csv("data/articles.csv", index=False, encoding="utf-8-sig")  # utf-8-sig เปิดได้ถูกใน Excel
print(f"\nSaved {len(df)} articles")
print(df[['title','char_count']].to_string())

### 02_ner_sentiment

In [ ]:
# 02_ner_sentiment.py
import pandas as pd
from transformers import pipeline
from collections import Counter

# โหลด model ครั้งเดียว
print("Loading NER model...")
ner = pipeline(
    task='ner',
    model='pythainlp/thainer-corpus-v2-base-model',
    aggregation_strategy='simple'
)
print("Model loaded!")

df = pd.read_csv(r"D:\Ma work\project\Information-Extraction-with-Small-Language-Model-and-Data-Analytics-Project\data\articles.csv", encoding="utf-8-sig")
print(f"Loaded {len(df)} articles")

# --- NER: ดึงเฉพาะ PERSON ---
def extract_persons(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    try:
        # model รับได้สูงสุด ~512 tokens — ตัด text ยาวเกินก่อน
        entities = ner(text[:1000])
        persons = [
            e['word'].strip()
            for e in entities
            if e['entity_group'] == 'PERSON'
            and len(e['word'].strip()) > 1
            and e['score'] > 0.85   # กรอง low confidence ออก
        ]
        return persons
    except Exception as ex:
        print(f"NER error: {ex}")
        return []

# --- Sentiment: rule-based เหมือนเดิม ---
POSITIVE = [
    'ชนะ', 'ประสบความสำเร็จ', 'เยี่ยม', 'โดดเด่น', 'ยกย่อง',
    'ชื่นชม', 'ภูมิใจ', 'สำเร็จ', 'ได้รับการยอมรับ', 'แก้ปัญหาได้',
    'ฟื้นตัว', 'ก้าวหน้า', 'บรรลุ', 'ยินดี', 'ดีใจ'
]
NEGATIVE = [
    'แพ้', 'วิจารณ์', 'โจมตี', 'เสียหาย', 'ล้มเหลว', 'ต่อต้าน',
    'ทุจริต', 'ฉ้อโกง', 'ถูกจับ', 'ลาออก', 'ไล่ออก', 'ประท้วง',
    'ขัดแย้ง', 'เสียชีวิต', 'อุบัติเหตุ', 'ระเบิด', 'โกง', 'คดี',
    'ระบาด', 'เตือน', 'อันตราย', 'วิกฤต', 'ถูกฟ้อง', 'จับกุม'
]

def get_sentiment(text):
    if not isinstance(text, str):
        return 'neutral', 0, 0
    pos = sum(1 for w in POSITIVE if w in text)
    neg = sum(1 for w in NEGATIVE if w in text)
    if pos > neg:
        return 'positive', pos, neg
    elif neg > pos:
        return 'negative', pos, neg
    return 'neutral', pos, neg

# --- ประมวลผล ---
print("Processing NER... (อาจใช้เวลา 2-3 นาที)")
df['persons']      = df['text'].apply(extract_persons)
df['person_count'] = df['persons'].apply(len)

sentiments         = df['text'].apply(get_sentiment)
df['sentiment']    = sentiments.apply(lambda x: x[0])
df['pos_score']    = sentiments.apply(lambda x: x[1])
df['neg_score']    = sentiments.apply(lambda x: x[2])

df.to_csv(r"D:\Ma work\project\Information-Extraction-with-Small-Language-Model-and-Data-Analytics-Project\data\articles_processed.csv", index=False, encoding="utf-8-sig")

# --- Summary ---
print("\n=== Results ===")
print(df[['title', 'sentiment', 'pos_score', 'neg_score', 'person_count']].to_string())

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

all_persons = [p for sublist in df['persons'] for p in sublist]
print("\nTop 10 persons mentioned:")
for name, count in Counter(all_persons).most_common(10):
    print(f"  {name}: {count}")